<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Módulo 1: Preprocesamiento y Representación de Texto</h1>
    <h2>Stopwords, Regex y Normalización</h2>
</div>


## 1. Stopwords 

### ¿Qué son las Stopwords?

Las **stopwords** son palabras de alta frecuencia y bajo contenido semántico que generalmente se eliminan antes del procesamiento: artículos, preposiciones, conjunciones y pronombres.

```
Oración:         "El gato de la casa come el pescado"
Sin stopwords:   ["gato", "casa", "come", "pescado"]
```

### Impacto en modelos NLP

| Modelo | ¿Eliminar stopwords? | Razón |
|---|---|---|
| TF-IDF / BoW | ✅ Sí | Reducen ruido y vocabulario |
| Naive Bayes | ✅ Sí | Mejora señal útil |
| Análisis de sentimientos | ⚠️ Depende | *"no me gustó"* — "no" es clave |
| BERT / Transformers | ❌ No | Aprende contexto directamente |
| Modelos de lenguaje | ❌ No | Necesitan texto completo |

In [ ]:
import re, string
from collections import Counter
import matplotlib.pyplot as plt

STOPWORDS_ES = {
    "a","al","algo","algunas","algunos","ante","antes","como","con",
    "contra","cual","cuando","de","del","desde","donde","durante",
    "e","el","ella","ellas","ellos","en","entre","era","eras","es",
    "esa","esas","ese","eso","esos","esta","estaba","estas","este",
    "esto","estos","fue","han","has","hay","la","las","le","les",
    "lo","los","me","mi","mis","muy","ni","no","nos","o","para",
    "pero","por","que","se","si","sin","sobre","su","sus","te",
    "también","tan","tanto","tu","tus","un","una","unos","unas",
    "y","ya","yo","más","él"
}

corpus_noticias = [
    "El gobierno de México anunció nuevas inversiones en tecnología e inteligencia artificial.",
    "La empresa Apple presentó su nuevo modelo de inteligencia artificial generativa.",
    "Los científicos descubrieron un nuevo método para el procesamiento de datos masivos.",
    "El banco central subió las tasas de interés para controlar la inflación.",
    "Investigadores de la UNAM desarrollaron un sistema de detección de enfermedades con IA.",
]

print("EFECTO DE ELIMINAR STOPWORDS:")
print("=" * 65)
for noticia in corpus_noticias[:2]:
    tokens = re.findall(r'[a-záéíóúüñ]+', noticia.lower())
    tokens_sin_stop = [t for t in tokens if t not in STOPWORDS_ES and len(t) > 2]
    print(f"\nNoticia: {noticia[:70]}")
    print(f"  Con stopwords:    {tokens[:10]}")
    print(f"  Sin stopwords:    {tokens_sin_stop}")
    print(f"  Reducción:        {len(tokens)} → {len(tokens_sin_stop)} tokens ({(1-len(tokens_sin_stop)/len(tokens))*100:.0f}% menos)")

In [ ]:
#  Stopwords con NLTK 
import nltk
from nltk.corpus import stopwords



In [ ]:
#  Visualización: palabras más frecuentes antes/después 
todos_tokens = []
for n in corpus_noticias:
    todos_tokens.extend(re.findall(r'[a-záéíóúüñ]+', n.lower()))

freq_con    = Counter(todos_tokens).most_common(15)
freq_sin    = Counter([t for t in todos_tokens if t not in stop_es and len(t) > 2]).most_common(15)


In [ ]:
freq_con

In [ ]:
freq_sin

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, data, titulo, color in [
    (axes[0], freq_con,  'Con stopwords', 'slategray'),
    (axes[1], freq_sin,  'Sin stopwords', 'steelblue')
]:
    words, counts = zip(*data)
    ax.barh(list(reversed(words)), list(reversed(counts)), color=color)
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel('Frecuencia')
plt.suptitle('Frecuencia de palabras en corpus de noticias', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Retomando las Expresiones Regulares para NLP 

### Patrones más usados en NLP

| Patrón | Descripción | Ejemplo |
|---|---|---|
| `\b[A-Z][a-z]+\b` | Palabras capitalizadas | Nombres propios |
| `\b[A-Z]{2,}\b` | Acrónimos | NLP, BERT, NASA |
| `\d+[.,]?\d*` | Números (enteros y decimales) | 3.14, 1000 |
| `\$\d+` | Precios | $500, $1,200 |
| `\d{1,2}/\d{1,2}/\d{2,4}` | Fechas | 21/05/2026 |
| `[\w.-]+@[\w.-]+\.\w+` | Correos electrónicos | user@mail.com |
| `https?://\S+` | URLs | https://spacy.io |
| `#\w+` | Hashtags | #NLP, #Python |
| `@\w+` | Menciones | @OpenAI |
| `\b\w{1,3}\b` | Palabras cortas (ruido) | "a", "es", "un" |

In [ ]:
import re

#  Extracción de patrones 
articulo = """
El 15/05/2026, la empresa OpenAI lanzó GPT-5 con un costo de $20 dólares mensuales.
El CEO Sam Altman anunció en Twitter (@sama): "La IA cambiará el mundo".
Para más información: https://openai.com/gpt5 o escribe a support@openai.com
Las acciones de NVIDIA subieron 15.3% mientras que META cayó un 3.2%.
#InteligenciaArtificial #GPT5 #IA
"""

patrones = {
    'Fechas':     r'\b\d{1,2}/\d{1,2}/\d{4}\b',
    'Precios':    r'\$\d+(?:[,.]\d+)?',
    'URLs':       r'https?://\S+',
    'Correos':    r'[\w.]+@[\w.]+\.[a-zA-Z]{2,}',
    'Hashtags':   r'#\w+',
    'Menciones':  r'@\w+',
    'Acrónimos':  r'\b[A-Z]{2,}\b',
    'Porcentajes':r'\d+\.?\d*%',
}

print("EXTRACCIÓN DE ENTIDADES CON REGEX:")
print("=" * 50)
for nombre, patron in patrones.items():
    matches = re.findall(patron, articulo)
    print(f"  {nombre:<15}: {matches}")

In [ ]:
# re.sub para limpieza avanzada 
def limpiar_con_regex(texto, opciones=None):
    """Aplica transformaciones de limpieza configurables."""
    if opciones is None:
        opciones = {"urls", "correos", "menciones", "hashtags",
                    "numeros", "emojis", "puntuacion", "espacios"}
    if "urls"       in opciones: texto = re.sub(r'https?://\S+', '', texto)
    if "correos"    in opciones: texto = re.sub(r'[\w.]+@[\w.]+\.[a-zA-Z]{2,}', '', texto)
    if "menciones"  in opciones: texto = re.sub(r'@\w+', '', texto)
    if "hashtags"   in opciones: texto = re.sub(r'#\w+', '', texto)
    if "numeros"    in opciones: texto = re.sub(r'\S*\d\S*', '', texto)
    if "emojis"     in opciones: texto = re.sub(r'[^\x00-\x7Fáéíóúüñ¡¿ÁÉÍÓÚÜÑa-zA-Z \n.,!?]', '', texto)
    if "puntuacion" in opciones: texto = re.sub(r'[^\w\sáéíóúüñÁÉÍÓÚÜÑ]', ' ', texto, flags=re.UNICODE)
    if "espacios"   in opciones: texto = re.sub(r'\s+', ' ', texto).strip()
    return texto.lower()


In [ ]:
textos_sucios = [
    "¡SÚPER oferta! 50% OFF 😍😍 https://tienda.mx #Descuento @TiendaMX",
    "Resultados del partido: 3-2 al min. 87 ⚽⚽⚽. Tremendo partido!",
    "Dr. Rodríguez (dr.rodriguez@hospital.mx) atendió a 45 pacientes hoy."
]


In [ ]:
#Limpieza con regex

## 3. Normalización de Texto 

La **normalización** convierte el texto a una forma estándar y consistente.

### Técnicas de normalización

| Técnica | Ejemplo | Uso |
|---|---|---|
| Minúsculas | `"NLP"` → `"nlp"` | Unificar variantes |
| Eliminar acentos | `"lección"` → `"leccion"` | Sistemas sin soporte unicode |
| Expandir contracciones | `"don't"` → `"do not"` | NLP en inglés |
| Normalizar espacios | `"hola   mundo"` → `"hola mundo"` | Limpieza básica |
| Eliminar caracteres repetidos | `"loooove"` → `"love"` | Texto informal |
| Normalizar números | `"cien"` → `"100"` | Estandarizar |
| Case folding | Unicode NFKD | Internacionalización |

#### Ejemplo simple de Normalización (Unicode)

In [ ]:
import unicodedata
import re

def quitar_acentos(texto):
    """Elimina tildes y diacríticos usando normalización Unicode."""
    nfkd = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

def normalizar_repeticiones(texto):
    """Reduce caracteres repetidos: loooove → love"""
    return re.sub(r'(.)\1{2,}', r'\1\1', texto)

def expandir_contracciones_en(texto):
    contracciones = {
        "don't": "do not", "won't": "will not", "can't": "cannot",
        "i'm": "i am", "i've": "i have", "it's": "it is",
        "he's": "he is", "she's": "she is", "they're": "they are",
        "we're": "we are", "you're": "you are", "i'd": "i would",
        "i'll": "i will", "isn't": "is not", "aren't": "are not",
    }
    texto = texto.lower()
    for contraccion, expansion in contracciones.items():
        texto = texto.replace(contraccion, expansion)
    return texto


In [ ]:
# Prueba
tests = [
    ("quitar_acentos",        "El niño comió en la lección de canción", quitar_acentos),
    ("normalizar_repeticiones","LOOOOOVE this soooong!!! Yesssss!!!",   normalizar_repeticiones),
    ("expandir_contracciones", "I don't think she's ready, won't try",  expandir_contracciones_en),
]



In [ ]:
# Normalización con unicodedata 

textos_unicode = [
    "Ñoño niño",
    "café résumé naïve",
    "ÑOÑO NIÑO",
    "Ação coração",     # Portugués
]


## 4. Pipeline de Limpieza Completo 

In [ ]:
import re, unicodedata, string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import pandas as pd

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

class PipelineTexto:
    """
    Pipeline completo de preprocesamiento NLP configurable.
    
    Etapas disponibles:
    1. Limpieza HTML/URLs/emojis
    2. Normalización (minúsculas, acentos, repeticiones)
    3. Tokenización
    4. Filtrado (stopwords, longitud)
    5. Stemming o lematización (opcional)
    """

    def __init__(self, idioma='spanish', quitar_acentos=False,
                 stemming=False, min_len=2):
        

    def _quitar_acentos(self, t):
        
        return None

    def paso1_limpiar(self, texto):
        # HTML tags <[^>]+>
        # URLs https?://\S+
        # Menciones @\w+
        # Hashtags #\w+
        # Números \S*\d\S*
        # Puntuación [^\w\sáéíóúüñÁÉÍÓÚÜÑ]
        
        return None

    def paso2_normalizar(self, texto):
       
        return None

    def paso3_tokenizar(self, texto):
        return None

    def paso4_filtrar(self, tokens):
        return None

    def paso5_normalizar_tokens(self, tokens):
        
        return None

    def procesar(self, texto, verbose=False):
        
        return None

print("✅ PipelineTexto definido con 5 etapas")

In [ ]:
# Prueba del pipeline con un ejemplo 
pipeline = PipelineTexto(idioma='spanish', stemming=False)

noticia = """¡ALERTA! El gobierno anunció HOY que invertirá $5000 millones 
en tecnología de inteligencia artificial 🤖. Según el presidente, 
esto creará empleos tecnológicos en todo el país. #IA #Tecnología
Más info en: https://gobierno.mx/tecnologia @GobMX"""


## 5. Ejercicio: Preprocesar Corpus de Noticias 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

#  Corpus de noticias simulado 
noticias = [
    {"categoria": "tecnología", "texto": "Apple presentó su nuevo chip M4 con IA integrada. La presentación fue en WWDC 2026 #Apple #M4"},
    {"categoria": "tecnología", "texto": "Google lanzó Gemini Ultra, su modelo más poderoso. Supera a GPT-4 en benchmarks https://blog.google.com"},
    {"categoria": "tecnología", "texto": "OpenAI anunció GPT-5 con capacidades multimodales avanzadas @OpenAI"},
    {"categoria": "economía",   "texto": "El peso mexicano se fortalece frente al dólar: $17.50 por USD en mercados internacionales"},
    {"categoria": "economía",   "texto": "La inflación en México cayó al 3.2% en abril 2026, según INEGI #Economia"},
    {"categoria": "economía",   "texto": "Inversión extranjera directa creció 15% en el primer trimestre del año"},
    {"categoria": "ciencia",    "texto": "Científicos descubrieron nueva especie de dinosaurio en Argentina #Paleontología"},
    {"categoria": "ciencia",    "texto": "Investigadores del MIT desarrollaron batería de estado sólido con carga en 5 minutos"},
    {"categoria": "ciencia",    "texto": "La NASA confirmó presencia de agua en la luna sur mediante misión Artemis https://nasa.gov"},
    {"categoria": "salud",      "texto": "Nuevo tratamiento contra el cáncer de páncreas muestra 80% de efectividad en ensayos clínicos"},
    {"categoria": "salud",      "texto": "OMS alerta sobre aumento de casos de resistencia a antibióticos en Latinoamérica #Salud"},
    {"categoria": "salud",      "texto": "Vacuna contra el dengue disponible en farmacias de México a partir de junio 2026"},
]

df = pd.DataFrame(noticias)
df

In [ ]:
#  Análisis del corpus procesado 


In [ ]:
# Palabras exclusivas por categoría
